# Table of Contents
* [1. Fast Generator](#generator)
* [2. Logic Visualizer](#visualizer)
* [3. Output Preview](#preview)

### 🧠 ARC Logic Engine

This tool decodes ARC-AGI patterns into human-readable rules to help you group similar tasks and design ONNX architectures faster.

---

### Usage
* **Group Tasks:** Find all "Symmetry" or "Rotation" tasks to reuse architectures.
* **Identify Difficulty:** See exactly where the LLM fails to prioritize manual work.
* **Architecture Mapping:** Use the logic to decide between Conv2D, Pooling, or Tiling layers.

### Next
* **Keyword-to-Operator Mapping:** Automatically link main rule words (e.g., "Rotate," "Mirror," "Scale") to specific low-cost ONNX operators to automate model design.
* **Semantic Embedding:** Use NLP embeddings to cluster tasks by rule similarity, allowing one optimized neural architecture to be reused for dozens of related problems.

---
### Dataset
To get the full data, you can visit my dataset produced from this notebook:
**[Logic for each ARC task](https://www.kaggle.com/datasets/karnakbaevarthur/logic-for-each-arc-task)**

---

## ⚡ Fast Generator <a id="generator"></a>

In [ ]:
import json, os, glob, time, csv
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# --- Configuration ---
START_TASK, END_TASK = 1, 10
BATCH_SIZE = 1  # Keeping it at 1 ensures the model focuses deeply on one specific logic
JSON_OUTPUT = '/kaggle/working/arc_explanations.json'
CSV_OUTPUT = '/kaggle/working/arc_explanations.csv'

# --- API Setup ---
user_secrets = UserSecretsClient()
client = OpenAI(
    api_key=user_secrets.get_secret("deepseek_api_key"),
    base_url="https://api.deepseek.com"
)

def format_grid(grid):
    return "\n".join([f"R{i}: {row}" for i, row in enumerate(grid)])

def save_dual_outputs(data_dict):
    with open(JSON_OUTPUT, 'w') as f:
        json.dump(data_dict, f, indent=2)
    with open(CSV_OUTPUT, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Task_ID', 'Short_Rule'])
        for tid in sorted(data_dict.keys()):
            writer.writerow([tid, data_dict[tid]])

# --- Engine ---
dataset_dir = '/kaggle/input/competitions/neurogolf-2026'
all_files = sorted(glob.glob(os.path.join(dataset_dir, "task*.json")))
explanations = {}

target_files = [f for f in all_files if START_TASK <= int(''.join(filter(str.isdigit, os.path.basename(f)))) <= END_TASK]

print(f"🚀 Processing {len(target_files)} tasks with Enhanced Prompting...")

for i in range(0, len(target_files), BATCH_SIZE):
    batch_files = target_files[i : i + BATCH_SIZE]
    
    # --- The "Logic First" Prompt ---
    prompt = """Analyze the transformation logic between the Input and Output grids. 
    Focus on: 
    1. Objects (groups of same-colored pixels) and their movement or scaling.
    2. Color changes (which color replaces which).
    3. Geometry (rotation, reflection, or symmetry).
    
    Provide a specific 2-3 sentence rule that works for ALL examples. Make rule specific'
    
    Return a JSON object where keys are Task IDs and values are the specific rule string."""
    
    for f in batch_files:
        tid = os.path.basename(f).replace('.json','')
        task_data = json.load(open(f))
        prompt += f"\n\n### TASK: {tid}\n"
        for idx, pair in enumerate(task_data['train']):
            prompt += f"--- Example {idx} ---\nIN:\n{format_grid(pair['input'])}\nOUT:\n{format_grid(pair['output'])}\n"

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": "You are a world-class ARC-AGI pattern recognizer. You identify exact spatial and logical transformations. Output ONLY valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                response_format={'type': 'json_object'}
            )
            
            batch_results = json.loads(response.choices[0].message.content)
            explanations.update(batch_results)
            save_dual_outputs(explanations)
            print(f"✅ Decoded: {list(batch_results.keys())}")
            break
        except Exception as e:
            print(f"⚠️ Retry: {e}")
            time.sleep(2)

print(f"🎉 Processed and saved to {JSON_OUTPUT} and {CSV_OUTPUT}")

## 📊 Output Preview <a id="preview"></a>

In [ ]:
import pandas as pd
import json

# 1. Display CSV shortly
print("📊 CSV PREVIEW (Short Rules):")
try:
    df = pd.read_csv('/kaggle/working/arc_explanations.csv')
    display(df.head(10)) # Shows first 10 rows
except Exception as e:
    print(f"CSV not found: {e}")

print("\n" + "="*50 + "\n")

# 2. Display JSON shortly
print("📄 JSON SNIPPET (First 3 entries):")
try:
    with open('/kaggle/working/arc_explanations.json', 'r') as f:
        data = json.load(f)
        # Get only the first 3 keys to keep it short
        short_data = {k: data[k] for k in list(data.keys())[:3]}
        print(json.dumps(short_data, indent=2))
except Exception as e:
    print(f"JSON not found: {e}")

## 🎨 Logic Visualizer <a id="visualizer"></a>

In [ ]:
import json, os, numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors

# --- Settings ---
START_VIS, END_VIS = 1, 10
JSON_INPUT = '/kaggle/working/arc_explanations.json'
DATASET_PATH = '/kaggle/input/competitions/neurogolf-2026' 

cmap = colors.ListedColormap(['#000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00', '#AAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'])
norm = colors.Normalize(vmin=0, vmax=9)

with open(JSON_INPUT, 'r') as f:
    explanations = json.load(f)

for n in range(START_VIS, END_VIS + 1):
    tid = f"task{n:03d}"
    fpath = os.path.join(DATASET_PATH, f"{tid}.json")
    if not os.path.exists(fpath): continue
        
    with open(fpath, 'r') as f: 
        task_data = json.load(f)
    
    # info is now a string because you stripped 'typical' from the generator
    rule = explanations.get(tid, "No rule found.")

    # Clean Console Print
    print(f"\n{'='*20} {tid} {'='*20}")
    print(f"RULE: {rule}\n")

    # Visuals
    train = task_data['train']
    fig, axes = plt.subplots(len(train), 2, figsize=(10, 3 * len(train)))
    if len(train) == 1: axes = [axes]
    
    # Update title to use the string directly
    plt.suptitle(f"{tid}: {rule[:70]}...", fontsize=11, fontweight='bold')

    for i, pair in enumerate(train):
        for j, key in enumerate(['input', 'output']):
            grid = np.array(pair[key])
            axes[i][j].imshow(grid, cmap=cmap, norm=norm)
            axes[i][j].set_title(f"{key.upper()} {grid.shape}", fontsize=9)
            axes[i][j].axis('off')
            
    plt.tight_layout()
    plt.show()

---
### Dataset
To get the full data, you can visit my dataset produced from this notebook:
**[Logic for each ARC task](https://www.kaggle.com/datasets/karnakbaevarthur/logic-for-each-arc-task)**

---

In [ ]:
import os
import sys
import json
import zipfile
import shutil
import re
import math
import tempfile
import importlib
import importlib.metadata
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# DEPENDENCY MANAGEMENT
# ─────────────────────────────────────────────────────────────────────────────
try:
    import onnx
except ImportError:
    os.system(f'{sys.executable} -m pip install onnx')
    import onnx

try:
    import onnxruntime as ort
except ImportError:
    os.system(f'{sys.executable} -m pip install onnxruntime')
    import onnxruntime as ort

def ensure_onnx_tool_v1():
    need_install = False
    try:
        if importlib.metadata.version('onnx-tool') != '1.0.0':
            need_install = True
    except importlib.metadata.PackageNotFoundError:
        need_install = True

    if need_install:
        print("  [*] Forcing installation of onnx-tool==1.0.0...")
        os.system(f'{sys.executable} -m pip install onnx-tool==1.0.0')
        if 'onnx_tool' in sys.modules:
            import onnx_tool
            importlib.reload(onnx_tool)

ensure_onnx_tool_v1()
import onnx_tool

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
TASK_PATTERN  = re.compile(r'^task\d{3}\.onnx$')
MAX_BYTES     = int(1.44 * 1024 * 1024)
BANNED_OPS    = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}

# ЧЕРНЫЙ СПИСОК: Задачи, которые ломают Kaggle Grader
BAD_TASKS = []
BLACKLIST = {f"task{str(t).zfill(3)}.onnx" for t in BAD_TASKS}

# Папки из вашего сообщения (скрипт сам найдет внутри zip-файлы)
SOLUTION_DIRS = {
    "aliafzal9323": Path('/kaggle/input/notebooks/artemnazemtsev/4275-submission'),
    "rauffauzanrambe": Path('/kaggle/input/notebooks/artemnazemtsev/neuro-golf-gambling-is-all-you-need'),
    "magmacot": Path('/kaggle/input/notebooks/jonathanchan/ngc36-constraint-smart-logic-mix-blending'),
}

PRIORITY = {
    "aliafzal9323": 1, 
    "rauffauzanrambe": 2, 
    "magmacot": 3,
}

OUT_ZIP = Path('./submission.zip')
OUT_DIR = Path('./submission_tasks')

# ─────────────────────────────────────────────────────────────────────────────
# FILE FINDING
# ─────────────────────────────────────────────────────────────────────────────
def load_from_zip(zip_path: Path) -> dict:
    graphs = {}
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for entry in zf.namelist():
                basename = os.path.basename(entry)
                if TASK_PATTERN.match(basename):
                    graphs[basename] = zf.read(entry)
    except Exception:
        pass
    return graphs

def load_graphs_from_dir(directory: Path, label: str) -> dict:
    graphs = {}
    if not directory.exists():
        print(f"  [!] [{label}] path not found: {directory}")
        return graphs
        
    if directory.is_file() and directory.suffix.lower() == '.zip':
        return load_from_zip(directory)

    # Ищем файлы и распаковываем зипы на лету
    for fpath in directory.rglob('*'):
        if not fpath.is_file(): continue
        
        if TASK_PATTERN.match(fpath.name):
            graphs[fpath.name] = fpath.read_bytes()
        elif fpath.suffix.lower() == '.zip':
            extracted = load_from_zip(fpath)
            if extracted:
                for k, v in extracted.items():
                    if k not in graphs:
                        graphs[k] = v
    return graphs

# ─────────────────────────────────────────────────────────────────────────────
# STATIC PROFILING & EXPLOIT CHECK (Fixed TypeError)
# ─────────────────────────────────────────────────────────────────────────────
def static_check(raw_bytes: bytes) -> tuple[bool, float, str]:
    if len(raw_bytes) > MAX_BYTES:
        return False, float('inf'), "Error: Size > 1.44MB"

    tmp_path = ""
    try:
        with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as tmp:
            tmp.write(raw_bytes)
            tmp_path = tmp.name

        try:
            model_proto = onnx.load(tmp_path)
        except Exception:
            return False, float('inf'), "Error: Bad ONNX structure"

        ops = {node.op_type for node in model_proto.graph.node}
        found_banned = BANNED_OPS & ops
        if found_banned:
            return False, float('inf'), f"Banned Ops: {found_banned}"

        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
        
        try:
            # Точная копия логики хоста Kaggle
            model = onnx_tool.loadmodel(tmp_path, {'verbose': False})
            g = model.graph
            
            try:
                g.graph_reorder_nodes()
            except:
                pass
                
            g.shape_infer(None)
            g.profile()
            
            # Проверка эксплойта отрицательной памяти
            if hasattr(g, 'nodemap'):
                for key in g.nodemap.keys():
                    if getattr(g.nodemap[key], 'memory', 0) < 0:
                        raise ValueError("Negative memory value detected")

            # ИСПРАВЛЕНИЕ ТУТ: sum(g.macs) так как g.macs это list!
            macs_list = getattr(g, 'macs', [0])
            macs = int(sum(macs_list)) if isinstance(macs_list, (list, tuple)) else int(macs_list)
            
            memory = int(getattr(g, 'memory', 0))
            params = int(getattr(g, 'params', 0))
            
            if memory < 0:
                 raise ValueError("Total memory is negative")

            cost = macs + memory + params
            
        finally:
            sys.stdout.close()
            sys.stdout = old_stdout
            
        os.remove(tmp_path)
        return True, max(1.0, float(cost)), "Success"

    except ValueError as ve:
        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
        return False, float('inf'), f"Exploit: {str(ve)}"
    except Exception as e:
        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
        import traceback
        return False, float('inf'), f"onnx-tool error: {type(e).__name__}"

# ─────────────────────────────────────────────────────────────────────────────
# MAIN ENSEMBLER LOGIC
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("Neurogolf Ensembler — Advanced Multi-Source Evaluator")
    print("=" * 60)

    loaded_sources = {}
    all_tasks_set = set()

    for label, dir_path in SOLUTION_DIRS.items():
        print(f"Scanning [{label}] ...")
        graphs = load_graphs_from_dir(dir_path, label)
        print(f"  → Found {len(graphs)} valid ONNX tasks")
        if graphs:
            loaded_sources[label] = graphs
            all_tasks_set.update(graphs.keys())

    if not loaded_sources:
        print("\nCRITICAL ERROR: No models found from any source.")
        sys.exit(1)

    all_tasks = sorted(all_tasks_set)
    best_graphs = {}
    sources = Counter()
    reasons = defaultdict(int)
    total_score = 0.0

    print(f"\nEnsembling {len(all_tasks)} unique tasks across {len(loaded_sources)} sources...")

    for task_key in all_tasks:
        if task_key in BLACKLIST:
            reasons['Blacklisted (Crashes Kaggle Grader)'] += 1
            continue
        candidates = []
        
        for label, store in loaded_sources.items():
            if task_key not in store: continue
            raw = store[task_key]
            
            ok_static, cost, fail_reason = static_check(raw)
            if not ok_static:
                reasons[f"{label} -> {fail_reason}"] += 1
                continue
                    
            candidates.append({
                'label': label,
                'cost': cost,
                'data': raw,
                'priority': PRIORITY.get(label, 99)
            })
            
        if candidates:
            candidates.sort(key=lambda x: (x['cost'], x['priority']))
            winner = candidates[0]
            best_graphs[task_key] = winner['data']
            sources[winner['label']] += 1
            total_score += max(1.0, 25.0 - math.log(winner['cost']) if winner['cost'] > 0 else 25.0)
        else:
            reasons['All sources failed this task'] += 1

    print("\n" + "=" * 60)
    print("ENSEMBLE RESULTS")
    print("=" * 60)
    print(f"Total Tasks Validated : {len(best_graphs)}")
    print(f"Total Tasks Skipped   : {len(all_tasks) - len(best_graphs)}")
    print(f"Predicted Score       : {total_score:,.2f}")
    print("-" * 60)

    if sources:
        print("Source Winning Split:")
        for src, count in sources.most_common():
            print(f"  - {src:<15}: {count} tasks")

    if reasons:
        print("\nDiagnostic Breakdown (Why models were rejected):")
        for reason, count in sorted(reasons.items()):
            print(f"  - {reason:<45}: {count}")

    if OUT_DIR.exists():
        shutil.rmtree(OUT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    if best_graphs:
        with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fname, fdata in best_graphs.items():
                (OUT_DIR / fname).write_bytes(fdata)
                zf.writestr(fname, fdata)
        print(f"\nSuccessfully saved {len(best_graphs)} tasks to {OUT_ZIP} and {OUT_DIR}/")
    else:
        print("\n[!] No tasks passed validation. Zip file was NOT created.")